# Week 31, Day 5: Spike timing: reliability and how well simple models predict

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w031/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Comparing spike trains

Write `coincidence(s1, s2, T, delta=2.0)`: count the spikes in s1 with a spike of s2 within delta ms, subtract the chance count 2 ν δ N1 (ν the rate of s2 per ms), normalise by half the total number of spikes and divide by 1 - 2 ν δ. Test it on two independent random trains of 40 spikes in 2 s.

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

def coincidence(s1, s2, T, delta=2.0):
    if len(s1) == 0 or len(s2) == 0:
        return 0.0
    n_coinc = ...
    nu = len(s2) / T
    return ...

r = np.random.default_rng(0)
a, b = np.sort(r.uniform(0, 2000, 40)), np.sort(r.uniform(0, 2000, 40))
g_random = coincidence(a, b, 2000)
print(f"identical: {coincidence(a, a, 2000):.2f};  independent: {g_random:.2f};  shifted by 1 ms: {coincidence(a, a + 1, 2000):.2f};  by 3 ms: {coincidence(a, a + 3, 2000):.2f}")

In [ ]:
import numpy as np
def test_identical():
    "identical trains give 1"
    s = np.array([10.0, 50, 90, 130])
    assert abs(coincidence(s, s, 200) - 1) < 1e-9

def test_random():
    "independent Poisson trains give a value near 0"
    assert abs(g_random) < 0.15

test_identical()
test_random()
print("All checks passed")

### Exercise: EIF against LIF

`eif(I, noise, seed, lif=False, Vth=None)` returns spike times. Measure `g_dc`, the mean coincidence between 6 trials with constant input, and `g_int`, the same for the frozen fluctuating input. Then compute `g_eif`, the mean coincidence of the noise-free EIF with each noisy trial, and `g_lif` for the LIF whose threshold (searched from -54 to -46 mV) best matches the mean spike count.

In [ ]:
dt, T = 0.05, 2000.0
n = int(T / dt)

def ou(mu, sd, tau, seed):
    r = np.random.default_rng(seed); x = np.empty(n); v = 0.0; a = math.exp(-dt / tau); xi = r.normal(size=n)
    for k in range(n):
        v = a * v + math.sqrt(1 - a * a) * xi[k]; x[k] = v
    return mu + sd * x

def eif(I, noise=0.0, seed=0, lif=False, Vth=None, C=200.0, gL=10.0, EL=-65.0, VT=-52.0, DT=1.5, Vr=-60.0):
    r = np.random.default_rng(seed); eta = r.normal(size=n) * noise / math.sqrt(dt)
    v, last, spikes = EL, -1e9, []
    for k in range(n):
        t = k * dt
        if t - last < 2.0:
            continue
        ex = 0.0 if lif else gL * DT * math.exp(min((v - VT) / DT, 20))
        v += dt * (-gL * (v - EL) + ex + I[k] + eta[k]) / C
        if (lif and v >= Vth) or (not lif and v > -30):
            spikes.append(t); last = t; v = Vr
    return np.array(spikes)

def mean_pairwise(trains):
    return np.mean([coincidence(trains[i], trains[j], T) for i in range(len(trains)) for j in range(len(trains)) if i != j])

I_dc = np.full(n, 200.0)
I_fl = ou(200.0, 150.0, 3.0, 11)
g_dc = mean_pairwise([eif(I_dc, 30.0, s) for s in range(6)])
trials = [eif(I_fl, 30.0, s) for s in range(6)]
g_int = ...
s_eif = eif(I_fl)
g_eif = ...
n_target = np.mean([len(t) for t in trials])
best = min(((abs(len(eif(I_fl, lif=True, Vth=v)) - n_target), v) for v in np.arange(-54, -46, 0.25)))
s_lif = eif(I_fl, lif=True, Vth=best[1])
g_lif = ...
print(f"constant input, trial to trial: {g_dc:.2f};  frozen noise, trial to trial: {g_int:.2f}")
print(f"noise-free EIF against the trials: {g_eif:.2f};  LIF (threshold {best[1]:.2f} mV): {g_lif:.2f}")

In [ ]:
import numpy as np
def test_dc():
    "with constant input the neuron's trials barely agree (below 0.3), with frozen noise they agree well (above 0.4)"
    assert g_dc < 0.3 and g_int > 0.4

def test_models():
    "the noise-free EIF predicts the trials at least as well as they predict each other, and the LIF does much worse"
    assert g_eif >= g_int - 0.05 and g_lif < g_eif - 0.2

test_dc()
test_models()
print("All checks passed")